# Perception Safety Batch Evaluation - Colab

Use this notebook to run YOLO batch inference on a nuScenes camera subset, project nuScenes 3D annotations into camera-image bounding boxes, calculate IoU/mAP metrics, and export CSV/Markdown artifacts for the Streamlit app.


In [ ]:
!pip install -q ultralytics opencv-python pandas numpy pillow tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Update this path to match your Google Drive folder.
NUSCENES_ROOT = Path('/content/drive/MyDrive/nuscenes_subset')
METADATA_ROOT = NUSCENES_ROOT / 'v1.0-trainval'
IMAGE_DIR = NUSCENES_ROOT / 'samples' / 'CAM_FRONT'
OUTPUT_DIR = Path('/content/drive/MyDrive/perception_eval_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Portfolio baseline: 'yolov8n.pt'. Stronger comparison: 'yolo11s.pt'.
MODEL_NAME = 'yolov8n.pt'
CONFIDENCE_THRESHOLD = 0.25
LOW_CONFIDENCE_THRESHOLD = 0.50
MAX_IMAGES = 500
RUN_ID = f"{MODEL_NAME.replace('.pt', '')}_conf{str(CONFIDENCE_THRESHOLD).replace('.', 'p')}"

print('Metadata root:', METADATA_ROOT)
print('Image dir:', IMAGE_DIR)
print('Output dir:', OUTPUT_DIR)
print('Run ID:', RUN_ID)
print('Image count:', len(list(IMAGE_DIR.glob('*.jpg'))) if IMAGE_DIR.exists() else 0)

In [ ]:
import json
import math
from collections import Counter, defaultdict

import numpy as np

IOU_THRESHOLD = 0.50
MAP_THRESHOLDS = [round(v / 100, 2) for v in range(50, 100, 5)]


def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)


def map_nuscenes_category_to_yolo(category_name):
    if category_name.startswith('human.pedestrian'):
        return 'person'
    if category_name == 'vehicle.car':
        return 'car'
    if category_name.startswith('vehicle.truck'):
        return 'truck'
    if category_name.startswith('vehicle.bus'):
        return 'bus'
    if category_name == 'vehicle.motorcycle':
        return 'motorcycle'
    if category_name == 'vehicle.bicycle':
        return 'bicycle'
    if category_name == 'movable_object.trafficcone':
        return 'traffic cone'
    return None


def quaternion_to_rotation_matrix(rotation):
    w, x, y, z = rotation
    norm = math.sqrt(w * w + x * x + y * y + z * z)
    if norm == 0:
        return np.eye(3)
    w, x, y, z = w / norm, x / norm, y / norm, z / norm
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)],
    ], dtype=float)


def box_3d_corners(center, size, rotation):
    width, length, height = size
    x_corners = [length / 2, length / 2, -length / 2, -length / 2, length / 2, length / 2, -length / 2, -length / 2]
    y_corners = [width / 2, -width / 2, -width / 2, width / 2, width / 2, -width / 2, -width / 2, width / 2]
    z_corners = [height / 2, height / 2, height / 2, height / 2, -height / 2, -height / 2, -height / 2, -height / 2]
    corners = np.array([x_corners, y_corners, z_corners], dtype=float)
    return quaternion_to_rotation_matrix(rotation) @ corners + np.array(center, dtype=float).reshape(3, 1)


def transform_points_to_camera(points_global, ego_pose, calibrated_sensor):
    points_ego = quaternion_to_rotation_matrix(ego_pose['rotation']).T @ (
        points_global - np.array(ego_pose['translation'], dtype=float).reshape(3, 1)
    )
    points_camera = quaternion_to_rotation_matrix(calibrated_sensor['rotation']).T @ (
        points_ego - np.array(calibrated_sensor['translation'], dtype=float).reshape(3, 1)
    )
    return points_camera


def project_camera_box_to_image(points_camera, camera_intrinsic, image_width, image_height):
    visible = points_camera[2, :] > 0.1
    if not visible.any():
        return None
    points = points_camera[:, visible]
    projected = np.array(camera_intrinsic, dtype=float) @ points
    projected = projected[:2, :] / projected[2:3, :]
    x1 = float(np.clip(projected[0, :].min(), 0, image_width))
    y1 = float(np.clip(projected[1, :].min(), 0, image_height))
    x2 = float(np.clip(projected[0, :].max(), 0, image_width))
    y2 = float(np.clip(projected[1, :].max(), 0, image_height))
    if x2 <= x1 or y2 <= y1:
        return None
    return (x1, y1, x2, y2)


def box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    intersection = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


def calculate_average_precision(detections, ground_truth_boxes, iou_threshold):
    gt_by_label = defaultdict(list)
    det_by_label = defaultdict(list)
    for gt in ground_truth_boxes:
        gt_by_label[gt['label']].append(gt)
    for det in detections:
        det_by_label[det['label']].append(det)

    aps = []
    for label, label_gt in gt_by_label.items():
        label_det = sorted(det_by_label.get(label, []), key=lambda row: row['confidence'], reverse=True)
        matched = set()
        tps = []
        fps = []
        for det in label_det:
            best_index = None
            best_iou = 0.0
            for index, gt in enumerate(label_gt):
                if index in matched:
                    continue
                iou = box_iou(det['bbox'], gt['bbox'])
                if iou > best_iou:
                    best_iou = iou
                    best_index = index
            if best_index is not None and best_iou >= iou_threshold:
                matched.add(best_index)
                tps.append(1)
                fps.append(0)
            else:
                tps.append(0)
                fps.append(1)

        cumulative_tp = 0
        cumulative_fp = 0
        recalls = [0.0]
        precisions = [1.0]
        for tp, fp in zip(tps, fps):
            cumulative_tp += tp
            cumulative_fp += fp
            recalls.append(cumulative_tp / len(label_gt))
            precisions.append(cumulative_tp / (cumulative_tp + cumulative_fp) if cumulative_tp + cumulative_fp else 0.0)
        recalls.append(1.0)
        precisions.append(0.0)
        for index in range(len(precisions) - 2, -1, -1):
            precisions[index] = max(precisions[index], precisions[index + 1])
        ap = 0.0
        for index in range(1, len(recalls)):
            if recalls[index] > recalls[index - 1]:
                ap += (recalls[index] - recalls[index - 1]) * precisions[index]
        aps.append(ap)

    return sum(aps) / len(aps) if aps else None


def calculate_iou_metrics(detections, ground_truth_boxes, iou_threshold=0.50):
    if not ground_truth_boxes:
        return {'matched': None, 'unmatched_detections': None, 'unmatched_ground_truth': None, 'mean_iou': None, 'iou_precision': None, 'iou_recall': None}
    matched_gt = set()
    matched_ious = []
    unmatched_detections = 0
    for det in sorted(detections, key=lambda row: row['confidence'], reverse=True):
        best_index = None
        best_iou = 0.0
        for index, gt in enumerate(ground_truth_boxes):
            if index in matched_gt or gt['label'] != det['label']:
                continue
            iou = box_iou(det['bbox'], gt['bbox'])
            if iou > best_iou:
                best_iou = iou
                best_index = index
        if best_index is not None and best_iou >= iou_threshold:
            matched_gt.add(best_index)
            matched_ious.append(best_iou)
        else:
            unmatched_detections += 1
    matched = len(matched_gt)
    return {
        'matched': matched,
        'unmatched_detections': unmatched_detections,
        'unmatched_ground_truth': len(ground_truth_boxes) - matched,
        'mean_iou': sum(matched_ious) / len(matched_ious) if matched_ious else None,
        'iou_precision': matched / len(detections) if detections else 0.0,
        'iou_recall': matched / len(ground_truth_boxes),
    }


def build_nuscenes_lookup(metadata_root):
    categories = load_json(metadata_root / 'category.json')
    instances = load_json(metadata_root / 'instance.json')
    annotations = load_json(metadata_root / 'sample_annotation.json')
    sample_data = load_json(metadata_root / 'sample_data.json')
    calibrated_sensors = load_json(metadata_root / 'calibrated_sensor.json')
    ego_pose_path = metadata_root / 'ego_pose.json'
    ego_poses = load_json(ego_pose_path) if ego_pose_path.exists() else []
    if not ego_poses:
        print('Warning: ego_pose.json not found. IoU/mAP box metrics will be N/A; count metrics still run.')

    category_by_token = {row['token']: row['name'] for row in categories}
    instance_category = {row['token']: category_by_token.get(row['category_token'], 'unknown') for row in instances}
    sample_data_by_image = {
        Path(row['filename']).name: row
        for row in sample_data
        if row.get('filename', '').startswith('samples/CAM_FRONT/') and row.get('is_key_frame')
    }
    calibrated_by_token = {row['token']: row for row in calibrated_sensors}
    ego_pose_by_token = {row['token']: row for row in ego_poses}

    expected = defaultdict(Counter)
    annotations_by_sample = defaultdict(list)
    for ann in annotations:
        category_name = instance_category.get(ann.get('instance_token', ''), 'unknown')
        label = map_nuscenes_category_to_yolo(category_name)
        if label:
            expected[ann['sample_token']][label] += 1
            annotations_by_sample[ann['sample_token']].append((label, ann))

    return {
        'expected_by_sample': {sample: dict(counts) for sample, counts in expected.items()},
        'sample_data_by_image': sample_data_by_image,
        'annotations_by_sample': annotations_by_sample,
        'calibrated_by_token': calibrated_by_token,
        'ego_pose_by_token': ego_pose_by_token,
    }


def get_ground_truth_boxes_for_image(image_path, lookup):
    sample_data = lookup['sample_data_by_image'].get(image_path.name)
    if not sample_data:
        return []
    calibrated_sensor = lookup['calibrated_by_token'].get(sample_data['calibrated_sensor_token'])
    ego_pose = lookup['ego_pose_by_token'].get(sample_data['ego_pose_token'])
    if calibrated_sensor is None or ego_pose is None:
        return []

    from PIL import Image
    with Image.open(image_path) as image:
        image_width, image_height = image.size

    boxes = []
    for label, ann in lookup['annotations_by_sample'].get(sample_data['sample_token'], []):
        corners_global = box_3d_corners(ann['translation'], ann['size'], ann['rotation'])
        corners_camera = transform_points_to_camera(corners_global, ego_pose, calibrated_sensor)
        bbox = project_camera_box_to_image(corners_camera, calibrated_sensor['camera_intrinsic'], image_width, image_height)
        if bbox:
            boxes.append({'label': label, 'bbox': bbox})
    return boxes


nuscenes_lookup = build_nuscenes_lookup(METADATA_ROOT)
expected_by_sample = nuscenes_lookup['expected_by_sample']
sample_data_by_image = nuscenes_lookup['sample_data_by_image']
print('Samples with expected counts:', len(expected_by_sample))
print('CAM_FRONT key frames:', len(sample_data_by_image))


In [ ]:
from ultralytics import YOLO
from tqdm import tqdm
import pandas as pd
import time

model = YOLO(MODEL_NAME)
image_paths = sorted(IMAGE_DIR.glob('*.jpg'))[:MAX_IMAGES]
rows = []
start_time = time.time()

for image_path in tqdm(image_paths):
    results = model.predict(str(image_path), conf=CONFIDENCE_THRESHOLD, verbose=False)
    result = results[0]
    for box in result.boxes:
        cls_id = int(box.cls.item())
        label = result.names[cls_id]
        confidence = float(box.conf.item())
        x1, y1, x2, y2 = [float(v) for v in box.xyxy[0].tolist()]
        rows.append({
            'image': image_path.name,
            'model': MODEL_NAME,
            'confidence_threshold': CONFIDENCE_THRESHOLD,
            'label': label,
            'confidence': confidence,
            'x1': x1,
            'y1': y1,
            'x2': x2,
            'y2': y2,
            'low_confidence': confidence < LOW_CONFIDENCE_THRESHOLD,
        })

elapsed = time.time() - start_time
detections_df = pd.DataFrame(rows)
detections_path = OUTPUT_DIR / f'perception_yolo_results_{RUN_ID}.csv'
detections_df.to_csv(detections_path, index=False)
print('Images evaluated:', len(image_paths))
print('Detections:', len(detections_df))
print('Elapsed seconds:', round(elapsed, 2))
print('Saved:', detections_path)
detections_df.head()

In [ ]:
summary_rows = []

for image_path in image_paths:
    image_name = image_path.name
    sample_data = sample_data_by_image.get(image_name)
    sample_token = sample_data.get('sample_token') if sample_data else None
    expected = Counter(expected_by_sample.get(sample_token, {}))
    image_detections = detections_df[detections_df['image'] == image_name] if not detections_df.empty else pd.DataFrame()
    detected = Counter(image_detections['label'].tolist()) if not image_detections.empty else Counter()
    detection_records = [
        {'label': row.label, 'confidence': row.confidence, 'bbox': (row.x1, row.y1, row.x2, row.y2)}
        for row in image_detections.itertuples()
    ] if not image_detections.empty else []
    ground_truth_boxes = get_ground_truth_boxes_for_image(image_path, nuscenes_lookup)

    labels = set(expected) | set(detected)
    true_positive = sum(min(expected[label], detected[label]) for label in labels)
    missed = {label: expected[label] - detected[label] for label in labels if expected[label] > detected[label]}
    false_pos = {label: detected[label] - expected[label] for label in labels if detected[label] > expected[label]}
    total_detected = sum(detected.values())
    total_expected = sum(expected.values())
    iou_metrics = calculate_iou_metrics(detection_records, ground_truth_boxes, IOU_THRESHOLD)
    map50 = calculate_average_precision(detection_records, ground_truth_boxes, 0.50)
    map_values = [
        value for value in [calculate_average_precision(detection_records, ground_truth_boxes, threshold) for threshold in MAP_THRESHOLDS]
        if value is not None
    ]
    map50_95 = sum(map_values) / len(map_values) if map_values else None

    summary_rows.append({
        'image': image_name,
        'sample_token': sample_token,
        'detected_total': total_detected,
        'expected_total': total_expected,
        'ground_truth_box_total': len(ground_truth_boxes),
        'missed_total': sum(missed.values()),
        'false_positive_total': sum(false_pos.values()),
        'low_confidence_total': int(image_detections['low_confidence'].sum()) if not image_detections.empty else 0,
        'precision': true_positive / total_detected if total_detected else 0.0,
        'recall': true_positive / total_expected if total_expected else 0.0,
        'iou_threshold': IOU_THRESHOLD,
        'iou_matched_detections': iou_metrics['matched'],
        'iou_unmatched_detections': iou_metrics['unmatched_detections'],
        'iou_unmatched_ground_truth': iou_metrics['unmatched_ground_truth'],
        'iou_precision': iou_metrics['iou_precision'],
        'iou_recall': iou_metrics['iou_recall'],
        'mean_iou': iou_metrics['mean_iou'],
        'map50': map50,
        'map50_95': map50_95,
        'expected_counts': dict(expected),
        'detected_counts': dict(detected),
        'missed_objects': missed,
        'false_positives': false_pos,
    })

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / f'perception_eval_summary_{RUN_ID}.csv'
summary_df.to_csv(summary_path, index=False)
print('Saved:', summary_path)
summary_df.head()


In [ ]:
report_path = OUTPUT_DIR / f'perception_failure_report_{RUN_ID}.md'
mean_precision = summary_df['precision'].mean() if not summary_df.empty else 0.0
mean_recall = summary_df['recall'].mean() if not summary_df.empty else 0.0
mean_iou = summary_df['mean_iou'].mean() if 'mean_iou' in summary_df and not summary_df['mean_iou'].dropna().empty else None
mean_map50 = summary_df['map50'].mean() if 'map50' in summary_df and not summary_df['map50'].dropna().empty else None
mean_map50_95 = summary_df['map50_95'].mean() if 'map50_95' in summary_df and not summary_df['map50_95'].dropna().empty else None

report = f"""# Perception Batch Evaluation Report

## Setup
- Model: {MODEL_NAME}
- Images evaluated: {len(image_paths)}
- Detection threshold: {CONFIDENCE_THRESHOLD}
- Low-confidence threshold: {LOW_CONFIDENCE_THRESHOLD}
- IoU threshold: {IOU_THRESHOLD}

## Aggregate Results
- Total detections: {len(detections_df)}
- Total expected objects: {int(summary_df['expected_total'].sum()) if not summary_df.empty else 0}
- Total projected ground-truth boxes: {int(summary_df['ground_truth_box_total'].sum()) if not summary_df.empty else 0}
- Total missed objects: {int(summary_df['missed_total'].sum()) if not summary_df.empty else 0}
- Total false positives: {int(summary_df['false_positive_total'].sum()) if not summary_df.empty else 0}
- Total low-confidence detections: {int(summary_df['low_confidence_total'].sum()) if not summary_df.empty else 0}
- Mean count precision: {mean_precision:.3f}
- Mean count recall: {mean_recall:.3f}
- Mean matched IoU: {'N/A' if mean_iou is None else f'{mean_iou:.3f}'}
- Mean mAP50: {'N/A' if mean_map50 is None else f'{mean_map50:.3f}'}
- Mean mAP50-95: {'N/A' if mean_map50_95 is None else f'{mean_map50_95:.3f}'}

## Safety Interpretation
- Missed vulnerable road users or vehicles should be treated as perception safety concerns for SOTIF and ISO 8800 follow-up.
- Low-confidence detections indicate candidates for threshold sensitivity analysis and scenario expansion.
- IoU and mAP metrics evaluate whether detections localize labeled objects, not only whether class counts match.
- False positives may cause unnecessary braking or planning disturbances.

## Recommended Follow-up
- Run the same notebook with `yolov8n.pt` and `yolo11s.pt` at confidence thresholds 0.25 and 0.50.
- Add camera channels beyond CAM_FRONT.
- Review low-IoU and unmatched boxes as formal perception failure examples.
- Import the CSV files into the Streamlit Project 3 dashboard.
"""

report_path.write_text(report)
print('Saved:', report_path)
print(report)
